In [1]:
from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageFile, ImageDraw, ImageOps

import matplotlib.pyplot as plt

from tqdm import tqdm

In [ ]:
POINT_COLUMNS = ["x1", "y1", "x2", "y2", "x3", "y3", "x4", "y4"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

In [ ]:
def load_image(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"OpenCV cannot decode {path}")
    return img


def normalized_points(row):
    pts = np.asarray([float(row[c]) for c in POINT_COLUMNS], dtype=np.float64).reshape(4, 2)
    if not np.all(np.isfinite(pts)):
        raise ValueError("NaN/Inf in OBB")
    if np.any(pts < 0) or np.any(pts > 1):
        raise ValueError(f"OBB outside [0,1]: {pts.tolist()}")
    return pts


def draw(img, pts_norm, label):
    h, w = img.shape[:2]
    pts = pts_norm.copy()
    pts[:, 0] *= w
    pts[:, 1] *= h
    q = np.round(pts).astype(np.int32).reshape(-1, 1, 2)
    cv2.polylines(img, [q], True, (0, 220, 0), max(2, int(max(h, w) / 1500)), cv2.LINE_AA)
    for i, (x, y) in enumerate(pts):
        cv2.circle(img, (int(x), int(y)), 5, (0, 0, 255), -1, cv2.LINE_AA)
        cv2.putText(img, str(i + 1), (int(x) + 7, int(y) - 7), cv2.FONT_HERSHEY_SIMPLEX, .6, (0, 0, 255), 2, cv2.LINE_AA)
    cv2.putText(img, label, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, .75, (255, 255, 255), 3, cv2.LINE_AA)
    cv2.putText(img, label, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, .75, (0, 0, 0), 1, cv2.LINE_AA)
    return img


def fit(img, max_side=1400):
    h, w = img.shape[:2]
    s = min(1.0, max_side / max(h, w))
    if s == 1:
        return img
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)

In [3]:
def make_sheet(files, output, cols=4, thumb_w=420):
    from PIL import ImageDraw
    thumbs = []
    for f in files:
        im = Image.open(f).convert("RGB")
        scale = thumb_w / im.width
        im = im.resize((thumb_w, max(1, int(im.height * scale))), Image.Resampling.LANCZOS)
        thumbs.append((f.name, im))
    if not thumbs:
        return
    cell_h = max(im.height for _, im in thumbs) + 30
    rows = math.ceil(len(thumbs) / cols)
    sheet = Image.new("RGB", (cols * thumb_w, rows * cell_h), "white")
    draw = ImageDraw.Draw(sheet)
    for i, (name, im) in enumerate(thumbs):
        x = (i % cols) * thumb_w
        y = (i // cols) * cell_h
        draw.text((x + 4, y + 3), name, fill="black")
        sheet.paste(im, (x, y + 24))
    output.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(output, quality=95)

In [ ]:
images = pd.read_csv("..\data\manifest\images.csv")[['image_id','raw_path']]
crops  = pd.read_csv("..\data\manifest\crops.csv")[['image_id','status','x1','y1','x2','y2','x3','y3','x4','y4']]
crops = crops[crops["status"].astype(str).eq("OK")].copy()
landmarks = pd.read_csv("..\data\manifest\landmarks_numbered.csv")[['image_id','status']]


<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Jules\AppData\Local\Temp\ipykernel_36824\2299620269.py:1: SyntaxWarning: invalid escape sequence '\d'
  images = pd.read_csv("..\data\manifest\images.csv")[['image_id','raw_path']]
C:\Users\Jules\AppData\Local\Temp\ipykernel_36824\2299620269.py:2: SyntaxWarning: invalid escape sequence '\d'
  crops  = pd.read_csv("..\data\manifest\crops.csv")[['image_id','status','x1','y1','x2','y2','x3','y3','x4','y4']]


In [14]:
df=crops.merge(images,on='image_id',how='left')
df=df.merge(landmarks,on='image_id',how='left').set_index('image_id')

In [ ]:
inds = [['b311167c4cd135b7','981597a6f2015adf','8250251205536775','4d55ad3b322d8eca',
         '5d77a635dfa000e8','8879c04cf52cbb1b','76a47e3bd3c67ec1','c1496842009c6707',
         'ada723da63f7c71a','1fafee4513da112a','59c50ca847ae3a27','4b658623405c06d4']]

rows = []
errors = []
for _, r in df.iterrows():
    try:
        src = Path(str(r["raw_path"]))
        with Image.open(src) as im:
            im.verify()
        pts = normalized_points(r)
        img = load_image(src)
        label = f"{r.image_id} | {r.specimen_id} | status=OK"
        img = fit(draw(img, pts, label))
    except Exception as e:
        errors.append({"image_id": r.get("image_id", ""), "error": repr(e)})